# Lesson 10: multidimensional random walks

This C++ notebook is the notebook form of **Example 10**. It shows how Marmote represents and simulates very large multidimensional random walks without storing a full transition matrix.

The model lives on a four-dimensional grid of size `1000 x 1000 x 100 x 100`, so the total state-space cardinal is `10^10`. The lesson compares two construction styles:

- through a generic `HomogeneousMultidTransition` object
- through the specialized random-walk classes provided by Marmote

As in the C++ example, we simulate both a continuous-time and a discrete-time chain. In the notebook, the simulation horizon is kept shorter than in the command-line example so that the printed trajectories remain readable.

**Import the modules**

In [1]:
// --- Marmote configuration for Xeus-cling ---
// These directives are technical and hidden from the rendered documentation.
#ifdef _WIN32
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include")
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include/marmoteCore")
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include/marmoteMarkovChain")
#pragma cling add_library_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/bin")
#pragma cling load("marmoteCore.dll")
#pragma cling load("marmoteMarkovChain.dll")
#else
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteCore")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteMarkovChain")
#pragma cling add_library_path("/home/assia/miniconda3/envs/xeus-cpp-env/lib")
#pragma cling load("libmarmoteCore.so")
#pragma cling load("libmarmoteMarkovChain.so")
#endif


The lesson uses the Markov chain simulation API together with the classes that encode homogeneous multidimensional transitions and random walks.

In [2]:
// --- Standard C++ utilities used in this notebook ---
#include <iomanip>
#include <iostream>

// --- Marmote headers used in this lesson ---
#include <marmoteMarkovChain/marmoteMarkovChain.h>
#include <marmoteMarkovChain/marmoteSimulationResult.h>
#include <marmoteMarkovChain/General/marmoteHomogeneousMultidBirthDeath.h>
#include <marmoteMarkovChain/General/marmoteHomogeneousMultidRandomWalk.h>
#include <marmoteCore/marmoteBox.h>
#include <marmoteCore/marmoteDiracDistribution.h>
#include <marmoteCore/marmoteDiscreteDistribution.h>
#include <marmoteCore/marmoteHomogeneousMultidTransition.h>

// --- Convenience declarations for the cells below ---
using namespace std;
using namespace marmote;


## Definition of the model

We first define a helper function for printing trajectories and the common model parameters used in all four phases of the example.

In [3]:
// Reprint the stored trajectory with the explicit multidimensional state.
void printTrajectory(SimulationResult* sr, MarmoteSet* sp)
{
    for (unsigned long int i = 0; i < sr->trajectorySize(); i++) {
        cout << setw(8) << i;
        cardinalType zestate = sr->states()[i];
        sp->PrintState(&cout, zestate);
        cout << endl;
    }
}

// The command-line example uses a very long horizon (1e4).
// In the notebook we use shorter runs to keep the output readable.
double tmax = 12.0;
simLenType nsteps = 12;

// Four-dimensional grid: 1000 x 1000 x 100 x 100.
stateType dims[4] = {1000, 1000, 100, 100};
double rateup[4] = {0.125, 0.125, 0.125, 0.125};
double ratedown[4] = {0.125, 0.125, 0.125, 0.125};

// Start all simulations from state (0, 0, 0, 0).
DiscreteDistribution* iDis = new DiracDistribution(0);


## Phase 1: continuous-time chain built from `HomogeneousMultidTransition`

We first build the transition structure explicitly, then wrap it in a generic `MarkovChain`. As in the original example, we check the consistency of the state-space cardinal in several objects before launching the simulation.

In [4]:
// Create the compact multidimensional transition structure in continuous time.
HomogeneousMultidTransition* genCT =
    new HomogeneousMultidTransition(CONTINUOUS, 4, dims, rateup, ratedown);
MarkovChain* mcCT = new MarkovChain(genCT);

// Check that all objects agree on the cardinal of the state space.
MarmoteSet* bigBox = new MarmoteBox(4, dims);
cout << "# Printing the size of the state space" << endl;
cout << "Big box cardinal           = " << bigBox->Cardinal() << endl;
cout << "HomMultidTrans size        = " << genCT->orig_size() << endl;
cout << "MultiDimHom space cardinal = "
     << genCT->orig_state_space()->Cardinal() << endl;
delete bigBox;

// Launch the continuous-time simulation.
mcCT->set_init_distribution(iDis);
cout << "# Execution of the continuous-time simulation with the trajectory printed along the way" << endl;
SimulationResult* srCT =
    mcCT->SimulateChainCT_AllOpt(tmax, false, true, false, true, true, CACHE_NONE);

// Print the stored trajectory again, but with the explicit state vector.
cout << "# Printing again the trajectory" << endl;
printTrajectory(srCT, genCT->orig_state_space());


# Printing the size of the state space
Big box cardinal           = 10000000000
HomMultidTrans size        = 10000000000
MultiDimHom space cardinal = 10000000000
# Execution of the continuous-time simulation with the trajectory printed along the way
[   0]     0.000000        0 (   0,   0,   0,   0)
[   1]     5.363037 10000000 (   1,   0,   0,   0)
[   2]     6.423999 10000001 (   1,   0,   0,   1)
[   3]     6.647652 20000001 (   2,   0,   0,   1)
[   4]     6.818022 20000000 (   2,   0,   0,   0)
[   5]     8.420709 20010000 (   2,   1,   0,   0)
[   6]     9.952623 20010100 (   2,   1,   1,   0)
[   7]    10.962698 20020100 (   2,   2,   1,   0)
[   8]    11.304501 20020200 (   2,   2,   2,   0)
[   9]    11.902307 20030200 (   2,   3,   2,   0)
[  10]    12.000000 30030200 (   3,   3,   2,   0)
# Printing again the trajectory
       0(   0,   0,   0,   0)
       1(   1,   0,   0,   0)
       2(   1,   0,   0,   1)
       3(   2,   0,   0,   1)
       4(   2,   0,   0,   0)
       

## Phase 2: discrete-time chain built from `HomogeneousMultidTransition`

We repeat the same construction in discrete time. The multidimensional transition object is reused as a compact representation of the random walk.

In [5]:
// Clean up the first experiment before starting the second one.
delete mcCT;
delete srCT;

// Create the compact multidimensional transition structure in discrete time.
HomogeneousMultidTransition* genDT =
    new HomogeneousMultidTransition(DISCRETE, 4, dims, rateup, ratedown);
MarkovChain* mcDT = new MarkovChain(genDT);

// Launch the discrete-time simulation.
mcDT->set_init_distribution(iDis);
cout << "# Execution of the discrete-time simulation with the trajectory printed along the way" << endl;
SimulationResult* srDT =
    mcDT->SimulateChainDT_AllOpt(nsteps, false, true, true, true, CACHE_NONE);

// Print the stored trajectory again.
cout << "# Printing again the trajectory" << endl;
printTrajectory(srDT, genDT->orig_state_space());


# Execution of the discrete-time simulation with the trajectory printed along the way
         0        0 (   0,   0,   0,   0)
         1        0 (   0,   0,   0,   0)
         2        0 (   0,   0,   0,   0)
         3        0 (   0,   0,   0,   0)
         4        0 (   0,   0,   0,   0)
         5        0 (   0,   0,   0,   0)
         6 10000000 (   1,   0,   0,   0)
         7 20000000 (   2,   0,   0,   0)
         8 20000100 (   2,   0,   1,   0)
         9 20010100 (   2,   1,   1,   0)
        10 30010100 (   3,   1,   1,   0)
        11 30000100 (   3,   0,   1,   0)
        12 30000100 (   3,   0,   1,   0)
# Printing again the trajectory
       0(   0,   0,   0,   0)
       1(   0,   0,   0,   0)
       2(   0,   0,   0,   0)
       3(   0,   0,   0,   0)
       4(   0,   0,   0,   0)
       5(   0,   0,   0,   0)
       6(   1,   0,   0,   0)
       7(   2,   0,   0,   0)
       8(   2,   0,   1,   0)
       9(   2,   1,   1,   0)
      10(   3,   1,   1,   0)
      

## Phase 3: direct continuous-time construction

Marmote also provides a specialized class for continuous-time multidimensional birth-death processes. This avoids building the generic transition object explicitly.

In [6]:
// Clean up the second experiment.
delete mcDT;
delete srDT;

// Build the same model directly as a multidimensional birth-death process.
MarkovChain* mcBirthDeath = new HomogeneousMultiDBirthDeath(4, dims, rateup, ratedown);
mcBirthDeath->set_init_distribution(iDis);

cout << "# Execution of the continuous-time simulation with the trajectory printed along the way" << endl;
SimulationResult* srBirthDeath =
    mcBirthDeath->SimulateChainCT_AllOpt(tmax, false, true, false, true, true, CACHE_NONE);

cout << "# Printing again the trajectory" << endl;
printTrajectory(srBirthDeath, mcBirthDeath->generator()->orig_state_space());


# Execution of the continuous-time simulation with the trajectory printed along the way
[   0]     0.000000        0 (   0,   0,   0,   0)
[   1]     3.150940      100 (   0,   0,   1,   0)
[   2]     4.669992      200 (   0,   0,   2,   0)
[   3]     4.873589      201 (   0,   0,   2,   1)
[   4]     7.423514 10000201 (   1,   0,   2,   1)
[   5]     7.703477 10000101 (   1,   0,   1,   1)
[   6]     8.136357 10000102 (   1,   0,   1,   2)
[   7]    10.029386 20000102 (   2,   0,   1,   2)
[   8]    10.750318 20010102 (   2,   1,   1,   2)
[   9]    12.000000 20000102 (   2,   0,   1,   2)
# Printing again the trajectory
       0(   0,   0,   0,   0)
       1(   0,   0,   1,   0)
       2(   0,   0,   2,   0)
       3(   0,   0,   2,   1)
       4(   1,   0,   2,   1)
       5(   1,   0,   1,   1)
       6(   1,   0,   1,   2)
       7(   2,   0,   1,   2)
       8(   2,   1,   1,   2)
       9(   2,   0,   1,   2)


## Phase 4: direct discrete-time construction

Finally, we use the specialized discrete-time random-walk class. The simulation logic is the same, but the model is created directly at the `MarkovChain` level.

In [7]:
// Clean up the third experiment.
delete mcBirthDeath;
delete srBirthDeath;

// Build the same model directly as a multidimensional random walk.
MarkovChain* mcRandomWalk = new HomogeneousMultiDRandomWalk(4, dims, rateup, ratedown);
mcRandomWalk->set_init_distribution(iDis);

cout << "# Execution of the discrete-time simulation with the trajectory printed along the way" << endl;
SimulationResult* srRandomWalk =
    mcRandomWalk->SimulateChainDT_AllOpt(nsteps, false, true, true, true, CACHE_NONE);

cout << "# Printing again the trajectory" << endl;
printTrajectory(srRandomWalk, mcRandomWalk->generator()->orig_state_space());


# Execution of the discrete-time simulation with the trajectory printed along the way
         0        0 (   0,   0,   0,   0)
         1      100 (   0,   0,   1,   0)
         2      101 (   0,   0,   1,   1)
         3      100 (   0,   0,   1,   0)
         4    10100 (   0,   1,   1,   0)
         5    10100 (   0,   1,   1,   0)
         6    10000 (   0,   1,   0,   0)
         7 10010000 (   1,   1,   0,   0)
         8 10010001 (   1,   1,   0,   1)
         9 10010002 (   1,   1,   0,   2)
        10 10010002 (   1,   1,   0,   2)
        11 10010102 (   1,   1,   1,   2)
        12 10010202 (   1,   1,   2,   2)
# Printing again the trajectory
       0(   0,   0,   0,   0)
       1(   0,   0,   1,   0)
       2(   0,   0,   1,   1)
       3(   0,   0,   1,   0)
       4(   0,   1,   1,   0)
       5(   0,   1,   1,   0)
       6(   0,   1,   0,   0)
       7(   1,   1,   0,   0)
       8(   1,   1,   0,   1)
       9(   1,   1,   0,   2)
      10(   1,   1,   0,   2)
      

## Cleanup

We release the objects created for the last experiment and the initial distribution shared by all simulations.

In [8]:
delete mcRandomWalk;
delete srRandomWalk;
delete iDis;
